# 🧠 Capability-Aware Specialist Model Exploration
## Strict Zero-Update Subnetwork Extraction ($\Delta \theta = 0, \theta_{\text{mini}} \subseteq \theta_{\text{teacher}}$)

This notebook demonstrates:
1. Loading standalone extracted models directly from Safetensors (`Specialist-1.0B`, `Specialist-1.4B`, `Base Teacher`).
2. Real-time side-by-side code generation and latency measurement.
3. Visualizing benchmark pass rates and capability density metrics ($NCD$).
4. Visualizing the $28 \times 28$ Layer Cosine Similarity Matrix.

In [1]:
import os
import time
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
print(f"Active Device: {device} | Dtype: {dtype}")

d:\conda_envs\py11_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active Device: cuda | Dtype: torch.bfloat16


### ⚡ 1. Load Standalone Specialist-1.0B (1.10B Parameters, -28.9% Sliced)

In [ ]:
model_path = "./outputs/specialist_1.0b_safetensors"

tokenizer = AutoTokenizer.from_pretrained(model_path)
specialist_1_0b = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None
)
specialist_1_0b.eval()

params = sum(p.numel() for p in specialist_1_0b.parameters())
print(f"Loaded Specialist-1.0B successfully! Parameters: {params / 1e6:.1f} M")

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 639.31it/s]


Loaded Specialist-1.0B successfully! Parameters: 725.9 M


### 🚀 2. Live Interactive Code Generation

In [3]:
prompt = "Write a Python function `fibonacci(n)` that returns the n-th Fibonacci number efficiently.\n```python\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

start_t = time.time()
with torch.no_grad():
    outputs = specialist_1_0b.generate(**inputs, max_new_tokens=64, do_sample=False)
elapsed = time.time() - start_t

gen_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]
print(f"Speed: {gen_tokens / elapsed:.1f} tokens/sec | Latency: {elapsed:.2f}s")
print("=" * 60)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("=" * 60)

Speed: 13.9 tokens/sec | Latency: 4.61s
Write a Python function `fibonacci(n)` that returns the n-th Fibonacci number efficiently.
```python
def fibonacci(n):
    if n < 1:
        raise ValueError("Input must be a positive integer")
    if n == 1:
        return 1
    if n == 2:
        return 1
    if n == 3:
        return 1



### 📊 3. Multi-Scale Benchmark Visualization (20 Algorithmic Questions)

In [ ]:
models = ["Base Teacher (1.54B)", "Specialist-1.4B (1.42B)", "Specialist-1.2B (1.26B)", "Specialist-1.0B (1.10B)"]
pass_rates = [5.0, 15.0, 35.0, 35.0]
latencies = [2.79, 2.82, 2.62, 2.11]
params_m = [1543.7, 1419.9, 1265.0, 1097.3]

fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:blue'
ax1.set_xlabel('Model Architecture', fontweight='bold')
ax1.set_ylabel('Pass Rate (%) on 20Q', color=color, fontweight='bold')
bars = ax1.bar(models, pass_rates, color=color, alpha=0.7, width=0.4)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 45)

for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Avg Latency (s/Q)', color=color, fontweight='bold')
ax2.plot(models, latencies, color=color, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(1.5, 3.5)

plt.title('20-Question Coding Benchmark: Pass Rate vs Latency (\Delta\theta = 0)', fontsize=13, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()